# SC-JEPA Training (Colab)

## 1. Clone repo

In [1]:
!git clone -b main https://github.com/Baptistecaille/SUPRACONDUCTOR-JEPA.git
%cd SUPRACONDUCTOR-JEPA

Cloning into 'SUPRACONDUCTOR-JEPA'...
remote: Enumerating objects: 233, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 233 (delta 0), reused 0 (delta 0), pack-reused 232 (from 2)
Receiving objects: 100% (233/233), 163.55 MiB | 17.04 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/SUPRACONDUCTOR-JEPA


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

In [8]:
import warnings
warnings.filterwarnings("ignore")

import torch
from pathlib import Path

CSV_PATH        = "data/jepa/mp.csv.gz"
CHECKPOINT_PATH = Path("/content/drive/MyDrive/SUPRA-JEPA/checkpoints/property_jepa_a100.pt")
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE   = 32
EPOCHS       = 5
MAX_BATCHES  = None   # set an int to cap batches per epoch
NUM_WORKERS  = 0

LAYERS       = 8
ATTN_HEADS   = 16
DROPOUT      = 0.0
EMA_DECAY    = 0.996

LR           = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
LOG_EVERY    = 1

print(f"Device: {DEVICE}")

Device: cuda


In [4]:
!pip install torch pymatgen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 138.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 130.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 12.8 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609 sha256=9a58e009fa5c4e5dff822de8dd37797f0ab335c0c568a1e3cfca2b16c7

## 3. Data

In [25]:
from crystal_matrices import create_dataloader

dataloader = create_dataloader(
    csv_path=CSV_PATH,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
print(f"Batches per epoch: {len(dataloader)}")

Batches per epoch: 2899


## 4. Model

In [26]:
from JEPA import (
    CrystalJEPA,
    CrystalTransformerEncoder,
    MaskConditionedPredictor,
    TargetCrystalTransformerEncoder,
)

context_encoder = CrystalTransformerEncoder(
    input_dim=103,
    layers=LAYERS,
    attn_heads=ATTN_HEADS,
    dropout=DROPOUT,
)
target_encoder = TargetCrystalTransformerEncoder(
    layers=LAYERS,
    attn_heads=ATTN_HEADS,
    dropout=DROPOUT,
)
predictor = MaskConditionedPredictor()

model = CrystalJEPA(
    context_encoder=context_encoder,
    target_encoder=target_encoder,
    predictor=predictor,
    ema_decay=EMA_DECAY,
).to(DEVICE)

model.update_ema(decay=0.0)  # copy context -> target at init

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

Parameters: 52,643,328


## 5. Optimizer

In [27]:
optimizer = torch.optim.AdamW(
    list(model.context_encoder.parameters()) + list(model.predictor.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

## 6. Training loop

In [28]:
def move_batch_to_device(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


global_step = 0

for epoch in range(EPOCHS):
    for batch_idx, batch in enumerate(dataloader):
        if MAX_BATCHES is not None and batch_idx >= MAX_BATCHES:
            break

        batch = move_batch_to_device(batch, DEVICE)

        model.train()
        optimizer.zero_grad(set_to_none=True)
        out = model(batch)
        loss = out["loss"]
        loss.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(
                list(model.context_encoder.parameters()) + list(model.predictor.parameters()),
                GRAD_CLIP,
            )
        optimizer.step()
        model.update_ema()

        global_step += 1
        if global_step % LOG_EVERY == 0:
            metrics = dict(out["metrics"])
            print(
                f"epoch={epoch} step={global_step} "
                f"loss={metrics['loss']:.6f} "
                f"loss_pred={metrics['loss_pred']:.6f}"
            )

print("Training complete.")

epoch=0 step=1 loss=3.490358 loss_pred=3.490358
epoch=0 step=2 loss=2.666806 loss_pred=2.666806
epoch=0 step=3 loss=2.296295 loss_pred=2.296295
epoch=0 step=4 loss=2.469707 loss_pred=2.469707
epoch=0 step=5 loss=2.365393 loss_pred=2.365393
epoch=0 step=6 loss=2.059333 loss_pred=2.059333
epoch=0 step=7 loss=2.170038 loss_pred=2.170038
epoch=0 step=8 loss=2.114305 loss_pred=2.114305
epoch=0 step=9 loss=1.780798 loss_pred=1.780798
epoch=0 step=10 loss=1.833282 loss_pred=1.833282
epoch=0 step=11 loss=2.031593 loss_pred=2.031593
epoch=0 step=12 loss=1.540424 loss_pred=1.540424
epoch=0 step=13 loss=1.568569 loss_pred=1.568569
epoch=0 step=14 loss=1.648295 loss_pred=1.648295
epoch=0 step=15 loss=1.903714 loss_pred=1.903714
epoch=0 step=16 loss=1.954853 loss_pred=1.954853
epoch=0 step=17 loss=1.676005 loss_pred=1.676005
epoch=0 step=18 loss=1.680773 loss_pred=1.680773
epoch=0 step=19 loss=1.704336 loss_pred=1.704336
epoch=0 step=20 loss=1.911961 loss_pred=1.911961
epoch=0 step=21 loss=1.774033

## 7. Save checkpoint

In [29]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "epoch": EPOCHS - 1,
        "step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    },
    CHECKPOINT_PATH,
)
print(f"Saved to {CHECKPOINT_PATH}")

Saved to /content/drive/MyDrive/SUPRA-JEPA/checkpoints/sc_jepa.pt


# Properties JEPA

In [7]:
!python -m models.property_jepa.training.train_property_jepa \
  --train-csv data/processed/property_jepa_train.csv.gz \
  --checkpoint-path checkpoints/property_jepa_a100.pt \
  --epochs 5 \
  --batch-size 256 \
  --device cuda \
  --regularizer vc \
  --reg-weight 0.01 \
  --scheduler cosine

/content/SUPRACONDUCTOR-JEPA/models/property_jepa/model/crystal_encoder.py:104: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=layers)
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3172: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3172: UserWarning: Issues encountered while parsing CIF: 12 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3172: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avo

In [12]:
!cp checkpoints/property_jepa_a100.pt /content/drive/MyDrive/SUPRA-JEPA/checkpoints/property_jepa_a100.pt

In [ ]:
class Value:
	def __init__(self, data, _children=(), _op=''):
		self.data = data
		self.grad = 0
		self._backward = lambda: None
		self._prev = set(_children)
		self._op = _op

	def __repr__(self):
		def fmt(x):
			return int(x) if float(x).is_integer() else round(x, 4)
		return f"Value(data={fmt(self.data)}, grad={fmt(self.grad)})"

	def __add__(self, other):
		other = other if isinstance(other, Value) else Value(other)
		out = Value(self.data + other.data, (self, other), '+')

		def _backward(self):
			out.grad += self.grad 
			out.grad += self.grad
		out._backward = _backward

		return out

	def __mul__(self, other):
		other = other if isinstance(other, Value) else Value(other)
		out = Value(self.data * other.data, (self, other), '*')

		def _backward(self):
			out.grad += self.data * self.grad 
			out.grad += self.other * self.grad
		out._backward = _backward

		return out

	def relu(self):
		out = Value(max(self.data, 0), (self,), "ReLU")

		def _backward(self):
			out.grad += 1.0 if self.data > 0 else 0.0

		out._backward = _backward
		return out

    def backward(self):
        
        tree = []
        nodes_visited = set()
        def build_tree(node):
            if node not in nodes_visited:
                nodes_visited.add(node)
                for child in node._prev:
                    build_tree(child)
                tree.append(node)
        build_tree(self)
        self.grad = 1.0
        for node in reversed(tree):
            node._backward()

		